# ICS 635 · Homework 2
## Fine-Tuning an LLM with LoRA

**Name:** TODO

Use this notebook to implement your own workflow. Read [ASSIGNMENT.pdf](ASSIGNMENT.pdf) for the
full requirements.

- Keep executable code, outputs, and written analysis here.
- Write concise analysis in Markdown cells beside the relevant results.
- Use `record.md` as optional scratch space; it does not need to be submitted.

Save your completed notebook with outputs.

## Setup

Follow `README.md` to set up the environment with `uv sync --locked` (or install packages manually yourself, listed in `pyproject.toml`).

Run the next three cells as they are. They keep the splits, the prompt format, and the scoring consistent, so that results can be compared.

In [1]:
import re

import torch
from datasets import concatenate_datasets, load_dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoTokenizer

SEED = 635 # DO NOT MODIFY
MODEL_NAME = "Qwen/Qwen3-1.7B"
ID_TO_LABEL = {0: "negative", 1: "positive", 2: "neutral"}
LABELS = list(ID_TO_LABEL.values())

device_type = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Device: {device_type}")

/var/home/sstevens/Documents/git/ics635-assignment2/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


In [2]:
data = load_dataset("zeroshot/twitter-financial-news-sentiment")
train = data["train"].shuffle(seed=SEED)
validation = data["validation"].shuffle(seed=SEED)

lora_train = train.select(range(3000))
example_pool = train.select(range(3000, len(train)))
selection = validation.select(range(600))
test = validation.select(range(600, len(validation)))
all_train = concatenate_datasets([lora_train, example_pool])

for name, part in [("lora_train", lora_train), ("example_pool", example_pool), ("selection", selection), ("test", test)]:
    print(f"{name}: {len(part):,} tweets")

Generating validation split: 100%|██████████| 2388/2388 [00:00<00:00, 222934.43 examples/s]

lora_train: 3,000 tweets
example_pool: 6,543 tweets
selection: 600 tweets
test: 1,788 tweets


In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def chat_prompt(content):
    """Wrap one user message in Qwen's chat format, with thinking turned off."""
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": content}],
        tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )


def parse_label(generated):
    """Return the first of "positive", "neutral" or "negative" in the text, ignoring case, or "unparsed"."""
    found = re.search(r"positive|neutral|negative", generated.lower())
    return found.group(0) if found else "unparsed"


def score(predictions, gold):
    """Score predicted label names against true label names. An unparsed output is always wrong."""
    per_class = f1_score(gold, predictions, labels=LABELS, average=None, zero_division=0)
    return {
        "macro_f1": round(float(per_class.mean()), 3),
        "accuracy": round(accuracy_score(gold, predictions), 3),
        **{f"f1_{label}": round(float(f1), 3) for label, f1 in zip(LABELS, per_class)},
        "unparsed": list(predictions).count("unparsed"),
    }


print(chat_prompt("Classify this tweet."))
print(parse_label("Neutral."), parse_label("I am not sure"))

<|im_start|>user
Classify this tweet.<|im_end|>
<|im_start|>assistant
<think>

</think>


neutral unparsed


## 1. Inspect the data

Summarize the labels in all four parts and read the tweets in `lora_train`. Calculate the scores of a method that always answers `neutral`.

**Finish this section:** Show the label table, at least five `lora_train` tweets per label, and the always-`neutral` scores on `selection`. In a Markdown cell below, explain what the label means, name two tweets you would have labeled differently, and explain why macro-F1 is the primary metric.

In [4]:
# TODO: Report the number of tweets and the share of each label in all four parts.
# TODO: Display at least five tweets for each label from lora_train.
# TODO: Score a method that always answers "neutral" on selection.

### Analysis

TODO: Explain what the label means, name two tweets you would have labeled differently and why, and explain why macro-F1 is the primary metric.

## 2. Establish baselines

Fit a `TfidfVectorizer` and `LogisticRegression(class_weight="balanced")` pipeline on `lora_train`, and again on `all_train`. Score both on `selection`.

**Finish this section:** Show both pipelines, their settings, and a table of scores and training times here. In a Markdown cell below, explain what each baseline tests and what the gap between them says.

In [5]:
# TODO: Fit both TF-IDF pipelines and measure their training times.
# TODO: Display a table of their selection scores.

### Analysis

TODO: Explain what each baseline tests and what the gap between them says about how much data matters to this method.

## 3. Prompt the base model

Load the base model and write batched generation code using the fixed settings in the assignment. Remember to pad on the left. Build every prompt with `chat_prompt`.

Write at least two zero-shot and at least four few-shot prompts. Take few-shot examples only from `example_pool`. Include two prompts with the same examples in a different order, and two with the same number of different examples.

**Finish this section:** Show the text of every prompt, with a placeholder where the tweet goes, and one table of `selection` results: macro-F1, accuracy, per-class F1, predicted-label counts, `unparsed` count, and time. Record your best zero-shot and best few-shot prompts, chosen on `selection` macro-F1.

In [6]:
# TODO: Load the base model and write a function that returns the generated text for a list of prompts.

In [7]:
# TODO: Define your prompts, score each one on selection, and display the results table.
# TODO: Record your best zero-shot prompt and your best few-shot prompt.

### Analysis

TODO: Explain which prompt changes mattered, what the order and different-example pairs show, and which two prompts you chose and why.

## 4. Fine-tune with LoRA

Train on `lora_train` with the fixed settings, using your best zero-shot prompt in each training example. Each example is the prompt, the label, and `<|im_end|>\n`. Remember to pad on the right for training and return to left padding for generation.

**Finish this section:** Show the configuration, one complete training example, the trainable parameter count and percentage, training time, peak GPU memory, adapter size, start and end loss, a plot of the training loss, and the `selection` scores of the model reloaded from its saved adapter.

In [8]:
# TODO: Build the training dataset and print one complete training example.

In [9]:
# TODO: Apply LoraConfig with get_peft_model and report the trainable parameters.
# TODO: Train with SFTTrainer and measure training time and peak memory.
# TODO: Plot the training loss from trainer.state.log_history.

In [10]:
# TODO: Save the adapter, release the model, and report the adapter size on disk.
# TODO: Reload it with AutoPeftModelForCausalLM and score it on selection.

### Analysis

TODO: Explain what LoRA trains and what stays frozen, why the adapter is so much smaller than the model, and what the loss curve shows.

## 5. Compare LoRA settings

Run the other four settings from the assignment's table, resetting the seed and loading a fresh base model each time. You already ran the first (`r=64`, `lora_alpha=32`) in Section 4. Save each adapter under its own name.

**Finish this section:** Show one table with all five runs (trainable parameters, adapter size, training time, peak memory, and `selection` scores) and a plot of macro-F1 against rank with two lines, one for `lora_alpha = r / 2` and one for `lora_alpha = 32`. The rank-64 point belongs on both lines, since 32 = 64 / 2. Record your final setting and the rule you used to choose it.

In [11]:
# TODO: Train and score the remaining settings. Release each model before loading the next.

In [12]:
# TODO: Display the table of all five runs and plot macro-F1 against rank.

### Analysis

TODO: Explain what rank and scaling each did to the score and the cost, which setting you chose and why, and how much a single seed and 600 tweets limit what you can conclude.

## Before final evaluation: record your final choices

Complete this checklist before proceeding to Section 6.

- [ ] Both TF-IDF pipelines are fitted and will not be changed.
- [ ] The best zero-shot and few-shot prompts are chosen and recorded.
- [ ] The final LoRA setting is chosen and its adapter is saved.
- [ ] Final prompts and the LoRA setting were chosen on `selection`; nothing used `test`.

| Method | Final configuration |
|---|---|
| TF-IDF on 3,000 tweets | TODO |
| TF-IDF on 9,543 tweets | TODO |
| Best zero-shot prompt | TODO |
| Best few-shot prompt | TODO |
| Final LoRA `r` and `lora_alpha` | TODO |

Do not revise these choices after inspecting test results.

## 6. Evaluate once on the test set

Score the six methods on `test` once each. Load the fine-tuned model from its saved adapter. Save the tweets, true labels, and every method's predictions as `test_predictions.csv`, and calculate the final metrics from that table.

**Finish this section:** Show one comparison table (macro-F1, accuracy, per-class F1, `unparsed` count, and prediction time per 1,000 tweets) and confusion matrices for the best zero-shot prompt and the fine-tuned model. Label every result as a test result.

In [13]:
# TODO: Generate test predictions for all six methods and measure prediction time.
# TODO: Save test_predictions.csv with one row per tweet.

In [14]:
# TODO: Calculate the scores and confusion matrices from test_predictions.csv, and add the prediction times measured above.

### Analysis

TODO: Compare the test results with selection, describe what changed between the two confusion matrices, and explain how the methods rank.

## 7. Analyze the failures

Read the errors. They come from `test`, so use them to explain your results and inform your recommendation, not to change any prompt or model.

**Finish this section:** Define at least three slices of `test`, and report the number of tweets and the macro-F1 of methods 3, 4 and 6 on each slice and on the rest of `test`. Find the tweets that methods 2 to 6 all got wrong, read at least 15 of them chosen with seed 635, mark each as mislabeled, ambiguous or hard with one line on why, and count the marks.

In [15]:
# TODO: Define at least three slices of test. For each slice and the rest of test, report the number of tweets and the macro-F1 of methods 3, 4 and 6.

In [16]:
# TODO: Find the tweets that methods 2 to 6 all got wrong and report how many there are.
# TODO: Read at least 15 of them, chosen with seed 635; mark each as mislabeled, ambiguous or hard, and count the marks.

### Analysis

TODO: Explain where fine-tuning's lead grows or shrinks across your slices and whether each slice is large enough to trust the difference, and what the tweets every method got wrong say about the labels.

### Recommendation

TODO: Someone needs to label this kind of tweet at scale. Using the scores, the training costs from Sections 2, 4 and 5, the prediction times from Section 6, and your error analysis, say which method you would choose and what would change your mind.